In [1]:
import numpy
import scipy.linalg

In [2]:
# Test data and reference solution

A = numpy.random.normal(size = (100, 25))
b = numpy.random.normal(size = 100)
x_ref, *_ = scipy.linalg.lstsq(A, b)

### Bonus task 13: gelss (easy)
Implement `gelss` (with numpy, python and scipy.svd).

Computes the minimum norm solution to a real linear least squares problem using the singular value decomposition (SVD).

$$ Ax = b $$
$$ U \Sigma V^T x = b $$
$$ x = (U \Sigma V^T) ^ {-1} b $$
$$ x = (V^T)^{-1} \Sigma ^ {-1} U^{-1} b $$
$$ x = V \dfrac{1}{\Sigma} U^T b $$

In [3]:
def gelss(A: numpy.ndarray, b: numpy.ndarray) -> numpy.ndarray:
    U, s, Vh = scipy.linalg.svd(A, full_matrices = False, lapack_driver = 'gesvd')
    return (Vh.T * (1 / s)) @ U.T @ b

assert ((gelss(A, b) - x_ref) < 1e-10).all()

### Bonus task 14: gelsd
Implement `gelsd` (with numpy and python ans scipy.svd).

The problem is solved in three steps:
- (1) Reduce the coefficient matrix A to bidiagonal form with
     Householder transformations, reducing the original problem
     into a "bidiagonal least squares problem" (BLS)
- (2) Solve the BLS using a divide and conquer approach.
- (3) Apply back all the Householder transformations to solve
     the original least squares problem.

In [4]:
# (1) Reduce the coefficient matrix A to bidiagonal form with Householder transformations

def get_transformation(x: numpy.ndarray, shape: int, start_from: int):
    u = x.copy()
    u[0] += numpy.sign(x[0]) * numpy.linalg.norm(x)
    u = u / numpy.linalg.norm(u)

    transformation = numpy.zeros((shape, shape))
    transformation[start_from:, start_from:] = u.reshape(-1, 1) @ u.reshape(1, -1)
    return numpy.eye(shape) - 2 * transformation

def nullify_column(matrix: numpy.ndarray, column: int, row_from: int):
    transformation = get_transformation(matrix[row_from:, column], matrix.shape[0], row_from)
    return transformation @ matrix, transformation

def nullify_row(matrix: numpy.ndarray, row: int, column_from: int):
    transformation = get_transformation(matrix[row, column_from:], matrix.shape[1], column_from)
    return matrix @ transformation, transformation

def reduce_matrix(matrix: numpy.ndarray):
    left_transformation = numpy.eye(matrix.shape[0])
    right_transformation = numpy.eye(matrix.shape[1])
    for i in range(min(*matrix.shape) - 1):
        matrix, transformation = nullify_column(matrix, i, i)
        left_transformation = transformation @ left_transformation

        matrix, transformation = nullify_row(matrix, i, i + 1)
        right_transformation = right_transformation @ transformation
    return matrix, left_transformation, right_transformation

# A small test
test_matrix = numpy.array([
    [ 1., 2, 3 ],
    [ 4, 5, 6 ],
    [ 7, 8, 15 ],
    [ 10, 9, 8 ]
])
transformed, left, right = reduce_matrix(test_matrix)
print(numpy.round(transformed, 3)) # It works

assert (left @ test_matrix @ right - transformed < 1e-10).all() # Direcft transformation
assert (numpy.linalg.inv(left) @ transformed @ numpy.linalg.inv(right) - test_matrix < 1e-10).all() # Inverse transformation

[[-12.884  20.995  -0.   ]
 [  0.      7.313   3.526]
 [  0.      0.     -1.148]
 [  0.     -0.     -0.066]]


In [5]:
def gelsd(A: numpy.ndarray, b: numpy.ndarray) -> numpy.ndarray:
    # (1) Reduce the coefficient matrix A to bidiagonal form with Householder transformations
    transformed, left, right = reduce_matrix(A)

    # (2) Solve the BLS using a divide and conquer approach.
    U, s, Vh = scipy.linalg.svd(transformed, full_matrices = False, lapack_driver = 'gesdd')
    x_transformed = (Vh.T * (1 / s)) @ U.T @ left @ b

    # (3) Apply back all the Householder transformations
    return right @ x_transformed 

assert ((gelsd(A, b) - x_ref) < 1e-10).all()

### Bonus task 15: gelsy
Implement `gelsy` (with numpy, python and scipy.qr).

The routine first computes a QR factorization with column pivoting:
```
     A * P = Q * [ R11 R12 ]
                 [  0  R22 ]
```
 with R11 defined as the largest leading submatrix whose estimated
 condition number is less than 1/RCOND.  The order of R11, RANK,
 is the effective rank of A.

 Then, R22 is considered to be negligible, and R12 is annihilated
 by orthogonal transformations from the right, arriving at the
 complete orthogonal factorization:
 ```
    A * P = Q * [ T11 0 ] * Z
                [  0  0 ]
```
 The minimum-norm solution is then
```
    X = P * Z**T [ inv(T11)*Q1**T*B ]
                 [        0         ]
```
 where Q1 consists of the first RANK columns of Q.

In [6]:
def gelsy(A: numpy.ndarray, b: numpy.ndarray, rcond: float = 1e-7):
    Q, R, P = scipy.linalg.qr(A, pivoting = True)

    diagonal = numpy.abs(numpy.diag(R))
    rank = numpy.sum(diagonal > rcond * diagonal.max())
    R[rank:, :] = numpy.zeros((R.shape[0] - rank, R.shape[1]))
    
    Z_T, T_T = scipy.linalg.qr(R.T)
    
    y = numpy.zeros(A.shape[1])
    y[:rank] = numpy.linalg.inv(T_T.T[:rank, :rank]) @ Q[:, :rank].T @ b

    x = numpy.zeros(A.shape[1])
    x[P, ...] = Z_T @ y
    return x

assert ((gelsy(A, b) - x_ref) < 1e-10).all()

### Bonus task 16: Cholesky
Fix bugs in the Cholesky decomposition algorithm.

In [7]:
A = numpy.random.normal(size=(5, 5))
A = A @ A.T + 1e-3 * numpy.eye(5)
numpy.linalg.cholesky(A) # True answer for reference

array([[ 2.26513498,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.11640396,  1.36167487,  0.        ,  0.        ,  0.        ],
       [-1.36776533,  0.22380788,  1.408868  ,  0.        ,  0.        ],
       [-1.19092474, -0.3077713 , -0.03671074,  1.83737882,  0.        ],
       [-1.2228095 , -1.14788532, -0.36787855, -0.21764899,  0.14550875]])

In [8]:
def cholesky(A: numpy.ndarray):
    L = numpy.zeros(A.shape)
    for i in range(A.shape[0]):
        for k in range(i + 1):
            s = numpy.dot(L[i, :k], L[k, :k])
            if i == k:
                if A[i, i] < s:
                    raise RuntimeError("Matrix is not positive definite")
                L[i, i] = numpy.sqrt(A[i, i] - s)
            else:
                L[i, k] = (A[i, k] - s) / L[k, k]
    return L

assert ((cholesky(A) - numpy.linalg.cholesky(A)) < 1e-10).all()